# ScratchFormer: Transformer from Scratch (Kaggle GPU Runner)

### Quick Settings Checklist on Kaggle:
1. **Accelerator**: Right sidebar → **Session options** → **Accelerator** → **GPU T4 x 2** (or GPU P100)
2. **Internet**: Right sidebar → **Session options** → **Internet** → **Turn ON** (needed to clone repo and download dataset)



## Cell 1: Environment & GPU Verification


In [ ]:
import os
import sys
import tensorflow as tf

# Check active GPU devices
!nvidia-smi

gpus = tf.config.list_physical_devices('GPU')
print(f'\nTensorFlow Version: {tf.__version__}')
if gpus:
    print(f'Active GPUs detected: {[gpu.name for gpu in gpus]}')
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
else:
    print('WARNING: No GPU detected! Please enable GPU T4 x 2 in the right sidebar under Session Options -> Accelerator.')

# Clone repository into /kaggle/working
%cd /kaggle/working
if not os.path.exists('ScratchFormer'):
    !git clone https://github.com/Ravikishore710/ScratchFormer.git

%cd /kaggle/working/ScratchFormer
!git pull origin main

if os.path.abspath('.') not in sys.path:
    sys.path.insert(0, os.path.abspath('.'))

print('Current Working Directory:', os.getcwd())



## Cell 2: Run Unit Test Suite (11 Core Invariant Tests)
Verifies mathematical correctness of masks, attention, and model gradients before starting training.


In [ ]:
!python -m pytest tests/ -v --tb=short


## Cell 3: Fast Pipeline Smoke Test
Validates data preparation, 1-epoch overfit check, greedy decoding, and attention plotting in ~1 minute.


In [ ]:
from src.config import Config
from src.data.dataset import prepare_datasets
from src.model.transformer import Transformer, count_parameters
from src.training.trainer import Trainer
from src.inference.generate import translate
import shutil

# Smoke data pipeline
b_smoke = prepare_datasets(Config(max_pairs=2000), verbose=True)

# Smoke model & overfit gate
cfg_smoke = Config(max_pairs=2000, epochs=1, batch_size=64)
m_smoke = Transformer(cfg_smoke, len(b_smoke.src_tok), len(b_smoke.tgt_tok))
print('Smoke Model Parameters:', count_parameters(m_smoke))
t_smoke = Trainer(m_smoke, cfg_smoke, checkpoint_dir='outputs/model/smoke')
t_smoke.overfit_check(b_smoke.train_ds, steps=50)
h_smoke = t_smoke.fit(b_smoke.train_ds, b_smoke.val_ds, log_every=10)

# Smoke translation
t_smoke.restore_latest()
print('Smoke Translation Sample:', translate(m_smoke, b_smoke.src_tok, b_smoke.tgt_tok, b_smoke.test_src_text[0]))

# Cleanup
shutil.rmtree('outputs/model/smoke', ignore_errors=True)
print('\nAll smoke tests PASSED! Ready for full training.')



## Cell 4: Full Baseline Training (60,000 Pairs, 10 Epochs)
On Kaggle GPU T4, each epoch takes only **~45–60 seconds**.
The full 10 epochs will complete in **under 10 minutes**!


In [ ]:
!python scripts/train.py --config configs/baseline.json --epochs 10


### Display Training & Validation Curves


In [ ]:
from IPython.display import Image, display

display(Image('outputs/training_curves/loss.png'))
display(Image('outputs/training_curves/acc.png'))



## Cell 5: Held-Out Test Set Evaluation & Predictions
Computes exact match %, corpus BLEU score, and sample translations for 500 test sentences.


In [ ]:
!python scripts/evaluate.py --config configs/baseline.json --n 500


In [ ]:
import json
import pandas as pd

with open('outputs/predictions/test_metrics.json') as f:
    metrics = json.load(f)
print('=== Test Evaluation Metrics ===')
print(json.dumps(metrics, indent=2))

df_preds = pd.read_csv('outputs/predictions/test_predictions.csv')
print('\n=== Sample Test Predictions (Top 10) ===')
display(df_preds[['source', 'reference', 'hypothesis', 'exact_match']].head(10))



## Cell 6: Attention Heatmap Visualizations
Generates and displays heatmaps for encoder self-attention, causal decoder self-attention, and cross-attention.


In [ ]:
!python scripts/visualize.py --config configs/baseline.json --n 3


In [ ]:
from IPython.display import Image, display

print('--- Positional Encoding ---')
display(Image('outputs/positional_encoding/positional_encoding.png'))

print('--- Encoder Self-Attention (Sample 0, Layer 0) ---')
display(Image('outputs/attention_maps/encoder/test_0_layer0.png'))

print('--- Decoder Causal Self-Attention (Sample 0, Layer 0) ---')
display(Image('outputs/attention_maps/decoder_self/test_0_layer0.png'))

print('--- Cross-Attention (Sample 0, Layer 0) ---')
display(Image('outputs/attention_maps/cross/test_0_layer0.png'))



## Cell 7: Full Ablation Grid & Keras-MHA Benchmark
Runs the 14 ablation experiments (PE, heads, width, depth, FFN, warmup, Keras MHA benchmark).


In [ ]:
!python scripts/run_experiments.py --config configs/baseline.json --epochs 3


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df_exp = pd.read_csv('outputs/experiments/experiment_results.csv')
print('=== Ablation Study Results Table ===')
display(df_exp)

# Size vs Quality Scatter Plot
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(df_exp['params'], df_exp['val_bleu'], c='royalblue', s=80, edgecolors='black', alpha=0.8)
for _, r in df_exp.iterrows():
    ax.annotate(r['experiment'], (r['params'], r['val_bleu']), fontsize=8, xytext=(4, 4), textcoords='offset points')
ax.set_xlabel('Trainable Parameters')
ax.set_ylabel('Validation BLEU')
ax.set_title('Size vs. Quality Trade-off across Ablations')
ax.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()



## Cell 8: Package & Download All Artifacts


In [ ]:
!zip -r /kaggle/working/scratchformer_outputs.zip outputs/
print('Output package saved to /kaggle/working/scratchformer_outputs.zip')
print('You can download it from the Output section in the right sidebar!')

